<a href="https://colab.research.google.com/github/manufloresg/TP1-escuelas-bibliotecas-argentinas/blob/main/notebooks/Infraestructura_Educacion_Argentina.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analisis de la infraestructura educativa de la Argentina

##1. Imports y configuracion

In [16]:
import pandas as pd
import duckdb as db
import numpy as np



##2. Carga de datos  

In [17]:
!git clone https://github.com/manufloresg/TP1-escuelas-bibliotecas-argentinas.git
%cd TP1-escuelas-bibliotecas-argentinas

bibliotecasPopulares = pd.read_csv("Data/tablas_originales/bibliotecas-populares.csv")
establecimientosEducativos = pd.read_csv("Data/tablas_originales/establecimientos_educativos.csv" ,encoding='ISO-8859-1',sep=';', skiprows=6)
establecimientosEducativos_original = establecimientosEducativos.copy()

padronPoblacion = pd.read_csv("Data/tablas_originales/padron_poblacion.xlsX - Output.csv")

Cloning into 'TP1-escuelas-bibliotecas-argentinas'...
remote: Enumerating objects: 53, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 53 (delta 20), reused 7 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (53/53), 3.83 MiB | 4.81 MiB/s, done.
Resolving deltas: 100% (20/20), done.
/content/TP1-escuelas-bibliotecas-argentinas/TP1-escuelas-bibliotecas-argentinas/TP1-escuelas-bibliotecas-argentinas/TP1-escuelas-bibliotecas-argentinas/TP1-escuelas-bibliotecas-argentinas/TP1-escuelas-bibliotecas-argentinas


##3. Modelo Relacional

In [18]:
provincia = (bibliotecasPopulares[["id_provincia", "provincia"]]
             .rename(columns={"provincia": "nombre"})
             .drop_duplicates()
             .sort_values("nombre")
             .reset_index(drop=True))

provincia

,id_provincia,nombre
0,6,Buenos Aires
1,10,Catamarca
2,22,Chaco
3,26,Chubut
4,2,Ciudad Autónoma de Buenos Aires
5,18,Corrientes
6,14,Córdoba
7,30,Entre Ríos
8,34,Formosa
9,38,Jujuy


In [19]:
departamento = (bibliotecasPopulares[["id_departamento", "departamento", "id_provincia"]]
                .rename(columns={"departamento": "nombre"})
                .dropna()
                .drop_duplicates()
                .reset_index(drop=True))
departamento

,id_departamento,nombre,id_provincia
0,6854,25 De Mayo,6
1,6588,9 De Julio,6
2,6007,Adolfo Alsina,6
3,6014,Adolfo Gonzalez Chaves,6
4,6021,Alberti,6
...,...,...,...
478,90021,Sin Partido,90
479,90112,Sin Partido,90
480,90098,Tafi Del Valle,90
481,90105,Tafi Viejo,90


In [20]:
biblioteca = (bibliotecasPopulares[["nro_conabip", "id_departamento", "fecha_fundacion", "mail"]]
              .rename(columns={"fecha_fundacion": "año_fundacion"})
              .drop_duplicates()
              .dropna()
              .reset_index(drop=True))

# Extraer solo el año — reemplaza la función soloAño original
biblioteca["año_fundacion"] = pd.to_datetime(biblioteca["año_fundacion"]).dt.year

biblioteca

,nro_conabip,id_departamento,año_fundacion,mail
0,1278,6007,1925,ccibiblio@hotmail.com
1,1062,6014,1925,bipobrchaves@yahoo.com.ar
2,2170,6014,1942,barrerabp@hotmail.com
3,1554,6021,1925,eleana_1983@hotmail.com
4,3792,6028,1980,bpmgandhi@tutopia.com
...,...,...,...,...
990,642,90070,1906,sociedadybibliotecamitre@yahoo.com.ar
991,2418,90070,1949,bibliotecaluisabuffodeferro@yahoo.com.ar
992,3116,90070,1987,biblioarancibia@hotmail.com
993,1799,90070,1929,bpcnv@yahoo.com.ar


In [21]:
establecimientosEducativos = establecimientosEducativos_original.copy()

establecimientosEducativos = establecimientosEducativos[
    establecimientosEducativos["Común"] == "1"
]

columnas_utiles = [
    "Código de localidad",
    "Cueanexo",
    "Nivel inicial - Jardín maternal",
    "Nivel inicial - Jardín de infantes",
    "Primario",
    "Secundario"
]

establecimientosEducativos = establecimientosEducativos[columnas_utiles].copy()

establecimientosEducativos["Inicial"] = np.where(
    (establecimientosEducativos["Nivel inicial - Jardín maternal"] == "1") |
    (establecimientosEducativos["Nivel inicial - Jardín de infantes"] == "1"),
    "1",
    ""
)

Escuelas = establecimientosEducativos.drop(
    columns=[
        "Nivel inicial - Jardín maternal",
        "Nivel inicial - Jardín de infantes"
    ]
).copy()

# Cueanexo como entero
Escuelas["Cueanexo"] = Escuelas["Cueanexo"].astype("Int64")

# Creo id_departamento con los primeros 5 dígitos reales del Código de localidad
Escuelas["id_departamento"] = (
    Escuelas["Código de localidad"]
    .astype("Int64")
    .astype(str)
    .str[:5]
)

# Elimino Código de localidad porque ya generé id_departamento
Escuelas = Escuelas.drop(columns=["Código de localidad"])
Escuelas.head()


,Cueanexo,Primario,Secundario,Inicial,id_departamento
0,20000100,1,1,1,21040
1,20000200,1,,,21040
2,20000300,1,,1,21130
3,20000500,1,1,1,21130
4,20000700,1,1,1,21090


In [22]:
col_edad = 'Unnamed: 1'
col_casos = 'Unnamed: 2'

es_area = padronPoblacion[col_edad].astype(str).str.startswith("AREA #", na=False)
indices_area = padronPoblacion[es_area].index.tolist()

dfs = []
saltados = 0

for i, idx in enumerate(indices_area):
    try:
        id_depto = int(
            str(padronPoblacion[col_edad].iloc[idx])
            .split("#")[1]
            .split()[0]
        )

        inicio = idx + 3
        fin = indices_area[i + 1] - 3 if i + 1 < len(indices_area) else len(padronPoblacion)

        casos = (
            padronPoblacion[col_casos]
            .iloc[inicio:fin]
            .dropna()
            .astype(str)
            .str.replace(" ", "", regex=False)
        )

        casos = pd.to_numeric(casos, errors="coerce").dropna().astype(int).reset_index(drop=True)

        if len(casos) < 19:
            saltados += 1
            continue

        dfs.append(pd.DataFrame({
            "id_departamento": [id_depto] * 4,
            "grupo_etario": ["0-5", "6-12", "12-18", ">18"],
            "cantidad": [
                casos.iloc[0:6].sum(),
                casos.iloc[6:12].sum(),
                casos.iloc[12:19].sum(),
                casos.iloc[19:].sum()
            ]
        }))

    except (ValueError, IndexError):
        saltados += 1

poblacion = pd.concat(dfs, ignore_index=True)

poblacion


,id_departamento,grupo_etario,cantidad
0,2007,0-5,12219
1,2007,6-12,14886
2,2007,12-18,17171
3,2007,>18,176724
4,2014,0-5,6616
...,...,...,...
2103,94011,>18,3951
2104,94015,0-5,6135
2105,94015,6-12,7604
2106,94015,12-18,8504


### Guardado de tablas

In [23]:
Escuelas.to_csv("Data/tablas_modelo/Escuelas.csv", index=False)
departamento.to_csv("Data/tablas_modelo/Departamentos.csv", index=False)
provincia.to_csv("Data/tablas_modelo/Provincias.csv", index=False)
biblioteca.to_csv("Data/tablas_modelo/Bibliotecas.csv", index=False)
poblacion.to_csv("Data/tablas_modelo/Poblacion.csv", index=False)

##4. Consultas SQL